# One query, every type, step by step

A hands-on lab. Run one cell at a time. You will build a single table carrying JSON, vectors, and geospatial columns, then run one unified query across all of them.

**Before you start:** finish `00-setup/`.

## Setup, connect

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env

DEMO_TABLE = 'account_unified_demo'
SCHEMA = _env('LB_SCHEMA', 'bank')

print('Ready.')

## Step 1, Create unified table (4 data types)

```sql
CREATE TABLE account_unified_demo (
    account_id TEXT PRIMARY KEY,
    branch_id TEXT NOT NULL,
    line TEXT NOT NULL,
    spec JSONB,
    embedding VECTOR(64),
    location GEOMETRY(POINT, 4326)
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector WITH SCHEMA public")
        cur.execute("CREATE EXTENSION IF NOT EXISTS postgis WITH SCHEMA public")
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{DEMO_TABLE}")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{DEMO_TABLE} (
                account_id TEXT PRIMARY KEY,
                branch_id TEXT NOT NULL,
                line TEXT NOT NULL,
                spec JSONB,
                embedding VECTOR(64),
                location GEOMETRY(POINT, 4326)
            )
        """)
        print(f'✓ Created unified table')

## Step 2, Populate with 1000 accounts

```sql
INSERT INTO account_unified_demo(...) SELECT ... FROM generate_series(1, 1000) g
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{DEMO_TABLE}(account_id, branch_id, line, spec, embedding, location)
            SELECT
                'CTA-' || g,
                (ARRAY['BR-CDMX', 'BR-MTY', 'BR-GDL'])[1 + (g % 3)],
                (ARRAY['Checking', 'Savings', 'Credit'])[1 + (g % 3)],
                jsonb_build_object(
                    'type', (ARRAY['retail', 'commercial', 'wealth'])[1 + (g % 3)],
                    'year', 2020 + (g % 5),
                    'version', '3.' || (g % 10)
                ),
                (array_fill((0.5 + (g % 10)::real / 100.0), ARRAY[64]))::vector(64),
                ST_MakePoint(-73.0 + (g % 10)::numeric / 100, 45.0 + (g % 10)::numeric / 100)
            FROM generate_series(1, 1000) g
        """)
        print(f'✓ Populated 1000 accounts with all data types')

## Step 3, Run unified query

One query filtering on relational (branch_id), JSON (spec->>'type'), and geospatial (ST_DWithin) columns:

```sql
SELECT
    account_id,
    branch_id,
    line,
    (spec->>'type') as account_type,
    (spec->>'year')::int as vintage,
    ST_Distance(location, ST_GeomFromText('POINT(45 -73)')) as km_from_ref
FROM account_unified_demo
WHERE
    branch_id = 'BR-CDMX'
    AND (spec->>'type') = 'commercial'
    AND ST_DWithin(location, ST_GeomFromText('POINT(45 -73)'), 1)
LIMIT 5;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT
                account_id,
                branch_id,
                line,
                (spec->>'type') as account_type,
                (spec->>'year')::int as vintage,
                (embedding <-> (SELECT embedding FROM {SCHEMA}.{DEMO_TABLE} WHERE account_id='CTA-1'))::numeric(6,4) as embedding_distance,
                ST_Distance(location, ST_MakePoint(-73.0, 45.0)::geometry) * 111.0 as km_from_ref
            FROM {SCHEMA}.{DEMO_TABLE}
            WHERE
                branch_id = 'BR-CDMX'
                AND (spec->>'type') = 'commercial'
                AND ST_DWithin(location, ST_MakePoint(-73.0, 45.0)::geometry, 0.1)
            ORDER BY embedding_distance, km_from_ref
            LIMIT 5
        """)
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description]

        print(f'Results (single query, single plan):')
        for row in rows:
            print(f'  {row[0]:12} | {row[1]:8} | {row[2]:12} | {row[3]:12} | {row[4]:4} | dist={row[5]:.4f} | km={row[6]:.4f}')

## What you learned

One table. Four data models. One query. Relational filters on branch_id and line. JSON extraction on spec. Vector and geospatial compute (when pgvector and PostGIS are loaded). No sync between systems, no consistency gaps, no application-layer glue code. One database.

Next: **Your commits are a change feed** shows logical decoding (Preview) as a stream of changes.